# Run Agent AutoML

Launch the AutoML agent loop for the resolved project. The agent creates project-local drafts and every accepted trial still runs through the same runner path used by human-authored models.

In [1]:
from __future__ import annotations

import shlex
import subprocess

from IPython.display import display

import automl
from automl import experiment

DRY_RUN = True
MAX_ITER = 1
AUTO_CONFIRM = True
OUTPUT_FORMAT = "text"
RUN_AGENT = True
BASE_SLUG = "xgboost example v2"
MODEL_INSTRUCTIONS = "train an xgboost model and learning from the past failure and improve from that"

In [2]:
active = automl.use_project(dry_run=DRY_RUN)
config = active.config
display(
    {
        "project": active.project_name,
        "repo_root": str(config.repo_root),
        "project_dir": str(config.project_dir),
        "experiment": active.active_experiment_id,
        "dry_run": active.dry_run,
    }
)

config_path = config.config_path
instructions_path = config.instructions_path
print(config_path)
print(config_path.read_text())
print(instructions_path)
print(instructions_path.read_text())


{'project': 'example_homecredit',
 'repo_root': '/Users/wendao/1.working_directory/brigit-data-science-automl-v1',
 'project_dir': '/Users/wendao/1.working_directory/brigit-data-science-automl-v1/projects/example_homecredit',
 'experiment': 'example-homecredit',
 'dry_run': True}

/Users/wendao/1.working_directory/brigit-data-science-automl-v1/projects/example_homecredit/config.py
"""Home Credit example project config for the four-layer refactor."""

from __future__ import annotations

from pathlib import Path

from automl.data import DataSpec, LocalCSVSource
from automl.eval import Auc, EvalSpec
from automl.model import RequiredTransformer
from automl.project import (
    BinaryClassification,
    ModelRoute,
    ModelsConfig,
    ProjectConfig,
    RunConfig,
    Splits,
)
from projects.example_homecredit.model.preprocessing import WOEEncoder


PROJECT_DIR = Path(__file__).resolve().parent
SAMPLE_CSV = PROJECT_DIR / "data" / "application_train_sample.csv"
HASH_KEY = "SK_ID_CURR"

TASK = BinaryClassification(target="TARGET")
DATA = DataSpec(
    source=LocalCSVSource(csv_path=SAMPLE_CSV, hash_key=HASH_KEY),
    metadata_cols=(HASH_KEY,),
    dry_run_rows=100,
)
EVAL = EvalSpec(primary=Auc())
RUN_CONFIG = RunConfig(
    experiment_id="example-homecredit",
    sp

In [3]:
def _slug_from_trial_id(trial_id: str) -> str:
    prefix, sep, rest = trial_id.partition("_")
    return rest if sep and prefix.isdigit() else trial_id


def existing_slugs() -> set[str]:
    slugs = set()
    experiments_dir = config.project_dir / "experiments"
    if experiments_dir.exists():
        slugs.update(path.name for path in experiments_dir.iterdir() if path.is_dir())
    leaderboard = experiment.leaderboard(training_origin="all", n=1000, session=active)
    for row in leaderboard.rows:
        if row.slug:
            slugs.add(_slug_from_trial_id(row.slug))
    return slugs


def next_available_slug(base_slug: str) -> str:
    used = existing_slugs()
    if base_slug not in used:
        return base_slug
    index = 2
    while f"{base_slug}_{index}" in used:
        index += 1
    return f"{base_slug}_{index}"


REQUESTED_SLUG = next_available_slug(BASE_SLUG)
RUN_INSTRUCTIONS = [
    f"use proposal slug exactly {REQUESTED_SLUG}",
    f"{MODEL_INSTRUCTIONS}",
]
display({"base_slug": BASE_SLUG, "requested_slug": REQUESTED_SLUG})

# Optional cleanup if you intentionally want to remove a prior run.
# from automl import trial
# cleanup_result = trial.delete("<run_id>", apply=False, session=active)
# display(cleanup_result)

command = [
    "uv",
    "run",
    "automl",
    "--project",
    active.project_name,
]
if DRY_RUN:
    command.append("--dry-run")
command.extend([
    "experiment",
    "run",
    "--output-format",
    OUTPUT_FORMAT,
])
if MAX_ITER is not None:
    command.extend(["--max-iter", str(MAX_ITER)])
if AUTO_CONFIRM:
    command.append("--auto-confirm")
for instruction in RUN_INSTRUCTIONS:
    command.extend(["--instruction", instruction])

print(" ".join(shlex.quote(part) for part in command))


{'base_slug': 'xgboost example v2', 'requested_slug': 'xgboost example v2'}

uv run automl --project example_homecredit --dry-run experiment run --output-format text --max-iter 1 --auto-confirm --instruction 'use proposal slug exactly xgboost example v2' --instruction 'train an xgboost model and learning from the past failure and improve from that'


In [4]:
if RUN_AGENT:
    subprocess.run(
        command,
        cwd=config.repo_root,
        stdin=subprocess.DEVNULL,
        check=True,
    )
else:
    print("Set RUN_AGENT = True in the command cell when you are ready to launch the agent loop.")


Execution error

Traceback (most recent call last):
  File "/Users/wendao/1.working_directory/brigit-data-science-automl-v1/automl/cli/__init__.py", line 36, in main
    finally:
             
  File "/Users/wendao/1.working_directory/brigit-data-science-automl-v1/automl/cli/_experiment_actions.py", line 39, in _run
    completed = subprocess.run(
                ^^^^^^^^^^^^^^^
  File "/Users/wendao/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/subprocess.py", line 550, in run
    stdout, stderr = process.communicate(input, timeout=timeout)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/wendao/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/subprocess.py", line 1201, in communicate
    self.wait()
  File "/Users/wendao/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/subprocess.py", line 1264, in wait
    return self._wait(timeout=timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/w

KeyboardInterrupt: 

In [ ]:
print("Outer launcher options: uv run automl --project example_homecredit experiment run --help")
print("Inner AutoML run options passed after the launcher options:")
print("  --max-iter N              stop after N AutoML iterations")
print("  --time-budget HOURS       session time budget")
print("  --auto-confirm            skip the run confirmation prompt")
print("  --refresh-data            rebuild active dataset")
print("  --refresh-source          refresh source inputs before dataset materialization")
print("  --instruction TEXT        one-off hard instruction for this run")
